# Prompt Engineering: Persona, Few-Shot, Negative Prompting, and CoT

> Actual API Calls !!!

**Tutorial reference:** Part 3 — Layer 2 of the SmartIntake Learning Tutorial

This notebook teaches you to design the system prompt that drives SmartIntake's extraction quality. You will build the prompt component by component, run ablation experiments to understand what each component contributes, and arrive at a production-quality prompt.

---

## 1. Setup

In [1]:
# Load environment variables from the .env file.
# python-dotenv reads the file and makes the key available via os.getenv().

from dotenv import load_dotenv
import os
import anthropic

load_dotenv()

# Load ANTHROPIC_API_KEY from the project-root .env (one folder up from week1/).
load_dotenv(dotenv_path=os.path.join('..', '.env'))

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not found. Add it to the project-root .env file.'
)

# The client reads ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

MODEL = 'claude-haiku-4-5'   # supports temperature / top_k / top_p (Opus 4.8 / 4.7 do not)
print(f'Client ready. Using model: {MODEL}')

Client ready. Using model: claude-haiku-4-5


In [ ]:
# Getting the function extract_with_prompt ready for use.
TEST_INPUTS = {
    "T1": (
        "PV team here. We have a serious unexpected SUSAR for study NCT-2244 "
        "and need to report to EMA within 15 days per ICH E2A."
    ),
    "T2": "Hi, we need to file a Type II variation for our cardiovascular product with the EMA.",
    "T3": (
        "CMC Regulatory here. FDA issued a 483 observation on our manufacturing site in Pune. "
        "We have 15 business days to respond."
    ),
    "T4": "Can you check what the weather is like in Mumbai today?",
    "T5": (
        "Labelling team. We need to update the SmPC for our oncology product "
        "following the latest EMA assessment report."
    ),
}


def extract_with_prompt(system_prompt: str, user_message: str) -> str:
    """Helper: call the API with a given system prompt and return the response text."""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text


print("Setup complete.")

Setup complete.


## 2. Baseline Prompt — No Engineering

Start with the bare minimum: a plain instruction with no persona, no examples, no constraints.

Run T1 and T5 through this baseline. Record the outputs. You will compare them against the engineered prompt later.

In [3]:
# Baseline: minimal instruction, no persona, no examples, no constraints.
# This is deliberately underspecified to illustrate the problem.

BASELINE_PROMPT = """
Extract five fields from the user's regulatory message:
query_type, regulation_ref, product_area, urgency, submitting_team.
Return as JSON.
"""

print("=== Baseline prompt results ===")
for key in ["T1", "T5"]:
    result = extract_with_prompt(BASELINE_PROMPT, TEST_INPUTS[key])
    print(f"\n{key}: {TEST_INPUTS[key][:80]}...")
    print(f"Result: {result}")

=== Baseline prompt results ===

T1: PV team here. We have a serious unexpected SUSAR for study NCT-2244 and need to ...
Result: ```json
{
  "query_type": "SUSAR Reporting",
  "regulation_ref": "ICH E2A",
  "product_area": "Clinical Study NCT-2244",
  "urgency": "High - 15-day reporting deadline",
  "submitting_team": "PV (Pharmacovigilance)"
}
```

T5: Labelling team. We need to update the SmPC for our oncology product following th...
Result: ```json
{
  "query_type": "SmPC Update",
  "regulation_ref": "EMA Assessment Report",
  "product_area": "Oncology",
  "urgency": "Not specified",
  "submitting_team": "Labelling"
}
```


## 3. Adding a Persona

A domain-specific persona shifts the model's prior toward regulatory terminology and reasoning patterns.

Compare the output of the persona-equipped prompt against the baseline for T1.

**Cline**: The key idea is that a persona prompt helps the model behave more like a domain expert, but it does not guarantee a different answer unless the task is tricky.



In [5]:
# Version 2: add a regulatory affairs persona.
# The persona tells the model how to approach the extraction, not just what to extract.

PERSONA_PROMPT = """
You are SmartIntake, a regulatory affairs intake specialist at a pharmaceutical company.
Your role is to classify incoming regulatory queries with precision.
You are familiar with FDA, EMA, ICH, and CDSCO regulatory frameworks.
You understand that urgency is deadline-driven, not sentiment-driven.

Extract these five fields from the user's message:
query_type, regulation_ref, product_area, urgency, submitting_team.
Return as a JSON object only. No additional text.
"""

print("=== Persona prompt results ===")
for key in ["T1", "T5"]:
    result = extract_with_prompt(PERSONA_PROMPT, TEST_INPUTS[key])
    print(f"\n{key}: {TEST_INPUTS[key][:80]}...")
    print(f"Result: {result}")

=== Persona prompt results ===

T1: PV team here. We have a serious unexpected SUSAR for study NCT-2244 and need to ...
Result: ```json
{
  "query_type": "SUSAR Reporting",
  "regulation_ref": "ICH E2A",
  "product_area": "Clinical Trial Safety",
  "urgency": "High - 15-day regulatory deadline",
  "submitting_team": "Pharmacovigilance (PV)"
}
```

T5: Labelling team. We need to update the SmPC for our oncology product following th...
Result: ```json
{
  "query_type": "Labelling Update",
  "regulation_ref": "EMA Assessment Report",
  "product_area": "Oncology",
  "urgency": "medium",
  "submitting_team": "Labelling"
}
```


## 4. Adding Few-Shot Examples

Few-shot examples show the model the exact format and calibration you expect. They are one of the most reliable techniques for extraction accuracy.

Use realistic pharmaceutical regulatory language in your examples — the same terminology a real regulatory affairs colleague would use.

In [6]:
# Version 3: persona + two few-shot examples.
# Each example uses authentic pharma regulatory language.
# Note the double braces {{ }} around JSON in f-strings to escape them.

FEW_SHOT_PROMPT = """
You are SmartIntake, a regulatory affairs intake specialist at a pharmaceutical company.
Your role is to classify incoming regulatory queries with precision.
You are familiar with FDA, EMA, ICH, and CDSCO regulatory frameworks.

Extract these five fields: query_type, regulation_ref, product_area, urgency, submitting_team.
Return ONLY a JSON object. No preamble, no explanation.

EXAMPLE 1:
Input: "We need to respond to an FDA query on our CMC section for NDA-209114."
Output: {"query_type": "submission", "regulation_ref": "FDA_21CFR", "product_area": "cmc", "urgency": null, "submitting_team": null}

EXAMPLE 2:
Input: "PV team here. We have a new serious unexpected SUSAR for the Phase III trial."
Output: {"query_type": "safety_signal", "regulation_ref": "ICH_E2A", "product_area": "clinical", "urgency": "critical", "submitting_team": "PV"}

For fields you cannot determine from the message, use null.
"""

print("=== Few-shot prompt results ===")
for key in ["T1", "T2", "T3"]:
    result = extract_with_prompt(FEW_SHOT_PROMPT, TEST_INPUTS[key])
    print(f"\n{key}: {TEST_INPUTS[key][:80]}...")
    print(f"Result: {result}")

=== Few-shot prompt results ===

T1: PV team here. We have a serious unexpected SUSAR for study NCT-2244 and need to ...
Result: {"query_type": "safety_signal", "regulation_ref": "ICH_E2A", "product_area": "clinical", "urgency": "critical", "submitting_team": "PV"}

T2: Hi, we need to file a Type II variation for our cardiovascular product with the ...
Result: {"query_type": "variation", "regulation_ref": "EMA_Reg1234_2008", "product_area": "cardiovascular", "urgency": null, "submitting_team": null}

T3: CMC Regulatory here. FDA issued a 483 observation on our manufacturing site in P...
Result: ```json
{"query_type": "regulatory_inspection", "regulation_ref": "FDA_21CFR", "product_area": "cmc", "urgency": "high", "submitting_team": "CMC Regulatory"}
```


## 5. Adding Negative Prompting

Negative prompting tells the model what not to do. The three critical constraints for SmartIntake address real failure modes:

1. Do not infer urgency from tone — ask for the actual deadline
2. Do not use an individual's name as `submitting_team`
3. Do not invent a `regulation_ref` — use `other` if unclear

In [7]:
# Version 4: persona + few-shot + negative prompting constraints.

NEGATIVE_PROMPT = """
You are SmartIntake, a regulatory affairs intake specialist at a pharmaceutical company.

Extract these five fields: query_type, regulation_ref, product_area, urgency, submitting_team.
Return ONLY a JSON object.

CRITICAL RULES — violations have compliance consequences:
1. NEVER infer urgency from tone. Words like 'please', 'ASAP', or 'urgent request' are not 
   evidence of a regulatory deadline. If the deadline is not explicit, set urgency to null.
2. NEVER use an individual person's name as submitting_team. Teams only (e.g. 'PV Team', 
   'CMC Regulatory'). If only a person's name is given, set submitting_team to null.
3. NEVER invent a regulation_ref. If the framework is not clearly identifiable, use 'other'.

EXAMPLE 1:
Input: "We need to respond to an FDA query on our CMC section for NDA-209114."
Output: {"query_type": "submission", "regulation_ref": "FDA_21CFR", "product_area": "cmc", "urgency": null, "submitting_team": null}

EXAMPLE 2:
Input: "PV team here. SUSAR for Phase III trial, EMA deadline 15 days, ICH E2A."
Output: {"query_type": "safety_signal", "regulation_ref": "ICH_E2A", "product_area": "clinical", "urgency": "critical", "submitting_team": "PV"}
"""

# Test the urgency constraint with a message that uses 'ASAP' without a real deadline
urgency_bait = (
    "Hi, can you please ASAP help us with a label update for our cardiovascular product? "
    "It's quite urgent. Thanks — regards, John from Labelling."
)

print("=== Urgency constraint test ===")
print(f"Input: {urgency_bait}")
print()

# Without negative prompt (baseline persona only)
result_without = extract_with_prompt(PERSONA_PROMPT, urgency_bait)
print(f"WITHOUT negative prompt: {result_without}")

print()

# With negative prompt
result_with = extract_with_prompt(NEGATIVE_PROMPT, urgency_bait)
print(f"WITH negative prompt: {result_with}")

=== Urgency constraint test ===
Input: Hi, can you please ASAP help us with a label update for our cardiovascular product? It's quite urgent. Thanks — regards, John from Labelling.

WITHOUT negative prompt: ```json
{
  "query_type": "Label Update",
  "regulation_ref": null,
  "product_area": "Cardiovascular",
  "urgency": "Low",
  "submitting_team": "Labelling"
}
```

WITH negative prompt: ```json
{
  "query_type": "labeling",
  "regulation_ref": "other",
  "product_area": "cardiovascular",
  "urgency": null,
  "submitting_team": null
}
```


## 6. Adding Chain-of-Thought Instruction

Adding "Think step by step about the regulatory context before choosing enum values" encourages the model to reason through the classification rather than pattern-matching to the first plausible answer.

This is most important for `urgency`, where the regulatory significance cannot always be inferred from surface language.

In [ ]:
# Version 5: complete production prompt — persona + few-shot + negative prompting + CoT.
# This is the prompt you will use in your SmartIntake implementation.

SMARTINTAKE_SYSTEM_PROMPT = """
You are SmartIntake, a regulatory affairs intake specialist at a pharmaceutical company.
Your role is to classify incoming regulatory queries with precision and professionalism.
You are expert in FDA, EMA, ICH, CDSCO, and GxP regulatory frameworks.

Think step by step about the regulatory context before choosing enum values.

Extract these five fields: query_type, regulation_ref, product_area, urgency, submitting_team.
Return ONLY a JSON object with these keys. No preamble. No explanation.

Allowed values:
  query_type: complaint | submission | variation | safety_signal | label_update | inspection | general_enquiry
  regulation_ref: FDA_21CFR | EMA_CTR | ICH_E2A | ICH_Q10 | CDSCO_NDC | GxP_GMP | GxP_GCP | other
  product_area: oncology | cardiovascular | infectious_disease | cmc | clinical | labelling | general
  urgency: routine | standard | urgent | critical
  submitting_team: <team or function name>

CRITICAL RULES:
1. NEVER infer urgency from tone. 'Please', 'ASAP', 'urgent request' are not regulatory deadlines.
   Only assign urgency if the message contains an explicit deadline or statutory timeframe.
   If unclear, set urgency to null.
2. NEVER use an individual person's name as submitting_team. If only a name is given, set null.
3. NEVER invent a regulation_ref. Use 'other' if the framework is not clearly identifiable.
4. If the query is not about pharmaceutical regulatory matters, return:
   {"error": "non_regulatory_input"}

EXAMPLE 1:
Input: "We need to respond to an FDA query on our CMC section for NDA-209114."
Output: {"query_type": "submission", "regulation_ref": "FDA_21CFR", "product_area": "cmc", "urgency": null, "submitting_team": null}

EXAMPLE 2:
Input: "PV team here. New SUSAR for Phase III trial. EMA deadline 15 days, ICH E2A."
Output: {"query_type": "safety_signal", "regulation_ref": "ICH_E2A", "product_area": "clinical", "urgency": "critical", "submitting_team": "PV"}
"""

print("=== Final production prompt — all five test inputs ===")
for key, message in TEST_INPUTS.items():
    result = extract_with_prompt(SMARTINTAKE_SYSTEM_PROMPT, message)
    print(f"\n{key}: {message[:80]}...")
    print(f"Result: {result}")

## 7. Ablation Study — Urgency Constraint

Remove the urgency constraint from the production prompt and run T1, T2, and T5.

Document how many times the model incorrectly infers `urgent` or `critical` from tone rather than asking for the actual deadline.

In [ ]:
# Ablation: remove rule 1 (the urgency constraint) from the production prompt.
# This isolates the contribution of that single constraint to extraction accuracy.

PROMPT_WITHOUT_URGENCY_RULE = SMARTINTAKE_SYSTEM_PROMPT.replace(
    "1. NEVER infer urgency from tone. 'Please', 'ASAP', 'urgent request' are not regulatory deadlines.\n"
    "   Only assign urgency if the message contains an explicit deadline or statutory timeframe.\n"
    "   If unclear, set urgency to null.\n",
    """
"""
)

# Test inputs where urgency inference from tone is a risk
tone_inputs = {
    "T2 (no deadline)": TEST_INPUTS["T2"],
    "T5 (no deadline)": TEST_INPUTS["T5"],
    "bait (ASAP + please)": (
        "URGENT please help asap — labelling team needs to update the SmPC immediately."
    ),
}

print("Urgency value with constraint vs without:\n")
print(f"{'Input':<30} {'WITH constraint':<25} {'WITHOUT constraint':<25}")
print("-" * 80)

for label, message in tone_inputs.items():
    with_result = extract_with_prompt(SMARTINTAKE_SYSTEM_PROMPT, message)
    without_result = extract_with_prompt(PROMPT_WITHOUT_URGENCY_RULE, message)

    # Extract just the urgency value for comparison
    def get_urgency(json_str):
        try:
            data = json.loads(json_str.strip().strip("```json").strip("```"))
            return str(data.get("urgency", "?"))
        except Exception:
            return "parse error"

    print(f"{label:<30} {get_urgency(with_result):<25} {get_urgency(without_result):<25}")

---
## Summary

You have:
- Built the SmartIntake system prompt component by component
- Observed the effect of each component through ablation experiments
- Confirmed that the urgency negative prompt constraint materially reduces tone-based urgency inference
- Produced the final `SMARTINTAKE_SYSTEM_PROMPT` that will be used in all subsequent notebooks

**Next:** Open `NB-04_function_calling_pydantic.ipynb` to enforce the schema through function calling and Pydantic validation.